# Evaluación de modelos de visión para GestCorp 98

**Pregunta:** ¿qué modelo sabe decirme *dónde* está un elemento de la interfaz (un botón, un campo
vacío…) a partir de su descripción?

**Cómo lo comparamos:** con 3 capturas de GestCorp 98 (`ground_truth/*.png`) y los puntos donde está
de verdad cada uno de los 7 elementos que buscamos (`ground_truth/alta_cliente.json`). Por cada
modelo y cada elemento se genera una imagen en `vision_evaluation/<modelo>/`:

- 🟢 **círculo verde** = dónde está el elemento de verdad
- 🔴 **círculo rojo** = dónde dice el modelo que está (si no hay rojo, el modelo no lo encontró)

La decisión se tomó **mirando esas imágenes**. Las conclusiones están al final del notebook.

Orden: **RapidOCR** (OCR, sin IA generativa) → **OmniParser** (detector YOLOv8 + Florence-2) →
**UI-TARS-7B-DPO** (modelo de visión hecho para agentes de interfaz).

## 0. Preparación

Carga los puntos buenos, define los 7 elementos a buscar (con la descripción que se le da a los
modelos que la entienden) y la función que dibuja los círculos, común a los tres modelos.

In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

CAPTURAS = Path("ground_truth")
SALIDA = Path("vision_evaluation")

# dónde está de verdad cada elemento, por captura: {"01_login.png": {"entrar_button": [x, y], ...}}
PUNTOS_BUENOS = json.loads((CAPTURAS / "alta_cliente.json").read_text(encoding="utf-8"))

# (captura, elemento, descripción para el modelo)
ELEMENTOS = [
    ("01_login.png", "usuario_field", "el campo de texto donde se escribe el nombre de usuario (ya contiene 'admin')"),
    ("01_login.png", "contraseña_field", "el campo de texto vacio donde se escribe la contraseña"),
    ("01_login.png", "entrar_button", "el boton 'Entrar' para iniciar sesion"),
    ("02_alta_cliente.png", "nombre_field", "el campo de texto vacio para escribir el nombre del cliente, en el formulario 'Alta de cliente'"),
    ("02_alta_cliente.png", "nif_field", "el campo de texto vacio para escribir el NIF del cliente"),
    ("02_alta_cliente.png", "anadir_button", "el boton 'Añadir cliente' para guardar el nuevo cliente"),
    ("03_popup_validacion.png", "popup_aceptar_button", "el boton 'Aceptar' del popup de aviso 'Datos incompletos'"),
]


def dibujar_comparacion(modelo: str, captura: str, elemento: str, punto_modelo) -> Path:
    """Guarda la captura con el punto bueno (verde) y el del modelo (rojo).
    `punto_modelo` es (x, y) en píxeles, o None si el modelo no lo encontró."""
    img = Image.open(CAPTURAS / captura).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    x, y = PUNTOS_BUENOS[captura][elemento]
    draw.ellipse((x - r, y - r, x + r, y + r), outline=(0, 200, 0), width=5)
    if punto_modelo is not None:
        px, py = punto_modelo
        draw.ellipse((px - r, py - r, px + r, py + r), outline=(255, 0, 0), width=5)
    carpeta = SALIDA / modelo
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta = carpeta / f"{Path(captura).stem}__{elemento}.png"
    img.save(ruta)
    print(f"{captura} / {elemento}: {'encontrado' if punto_modelo else 'NO encontrado'} -> {ruta}")
    return ruta


print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. RapidOCR

OCR puro: detecta **todo el texto** de la captura con su caja. No entiende descripciones, así que
para cada elemento buscamos un texto que lo identifique:

- **Botones:** el texto está *dentro* del botón ("Entrar", "Aceptar"), así que su caja es el botón.
- **Campos vacíos:** dentro no hay texto que leer. Lo más cercano es su etiqueta ("Usuario:",
  "NIF:"), que está *al lado* del campo, no encima.

In [ ]:
import unicodedata

from rapidocr_onnxruntime import RapidOCR

_ocr = RapidOCR()


def ask_rapidocr(captura: str) -> list[dict]:
    """Todos los textos que RapidOCR encuentra en la captura, con su caja."""
    resultado, _ = _ocr(str(CAPTURAS / captura))
    return [{"texto": texto, "caja": caja} for caja, texto, _conf in (resultado or [])]


def _normalizar(texto: str) -> str:
    """Minúsculas y sin tildes, para comparar sin depender de cómo transcribe el OCR."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii").lower().strip()


# texto que buscamos para cada elemento (en los campos, su etiqueta)
TEXTO_OCR = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre:",
    "nif_field": "nif:",
    "anadir_button": "anadir cliente",
    "popup_aceptar_button": "aceptar",
}


def punto_rapidocr(detecciones: list[dict], texto: str):
    """Centro de la primera caja cuyo texto contiene `texto`, o None."""
    for d in detecciones:
        if texto in _normalizar(d["texto"]):
            xs = [p[0] for p in d["caja"]]
            ys = [p[1] for p in d["caja"]]
            return sum(xs) / len(xs), sum(ys) / len(ys)
    return None

In [ ]:
# prueba rápida: todo lo que lee en el formulario de alta
for d in ask_rapidocr("02_alta_cliente.png"):
    print(d["texto"])

In [ ]:
# imágenes etiquetadas -> vision_evaluation/rapidocr/
detecciones = {c: ask_rapidocr(c) for c in dict.fromkeys(e[0] for e in ELEMENTOS)}
for captura, elemento, _desc in ELEMENTOS:
    dibujar_comparacion("rapidocr", captura, elemento,
                        punto_rapidocr(detecciones[captura], TEXTO_OCR[elemento]))

## 2. OmniParser

`microsoft/OmniParser-v2.0` tiene dos piezas:

- un **detector YOLOv8** que encuentra todas las cajas de elementos de la interfaz;
- **Florence-2** como *captioner*, que describe en texto qué es cada caja.

Primero dibujamos **todas las cajas** que detecta, para ver qué tal localiza. Después intentamos
quedarnos con la caja de cada elemento buscando su nombre en la descripción de Florence-2.

> Detalle de carga: el repo de OmniParser solo trae los pesos afinados del captioner. Su procesador
> se carga del modelo base, `microsoft/Florence-2-base`, como hace el `handler.py` oficial.

In [ ]:
_detector = _captioner = _procesador = None


def _cargar_omniparser():
    global _detector, _captioner, _procesador
    if _detector is None:
        from huggingface_hub import snapshot_download
        from transformers import AutoModelForCausalLM, AutoProcessor
        from ultralytics import YOLO

        carpeta = snapshot_download("microsoft/OmniParser-v2.0")  # ~1.4 GB la primera vez
        _detector = YOLO(f"{carpeta}/icon_detect/model.pt")
        _procesador = AutoProcessor.from_pretrained("microsoft/Florence-2-base", trust_remote_code=True)
        _captioner = AutoModelForCausalLM.from_pretrained(
            f"{carpeta}/icon_caption", torch_dtype=torch.float16, trust_remote_code=True
        ).to("cuda")
    return _detector, _captioner, _procesador


def ask_omniparser(captura: str) -> list[dict]:
    """Todas las cajas que detecta, cada una con la descripción de Florence-2."""
    detector, captioner, procesador = _cargar_omniparser()
    imagen = Image.open(CAPTURAS / captura).convert("RGB")
    cajas = [box.xyxy[0].tolist() for box in detector(str(CAPTURAS / captura), verbose=False)[0].boxes]
    if not cajas:
        return []
    # recortes a 64x64, como en el handler oficial
    recortes = [imagen.crop(tuple(c)).resize((64, 64)) for c in cajas]
    entradas = procesador(
        images=recortes, text=["<CAPTION>"] * len(recortes), return_tensors="pt", do_resize=False
    ).to(captioner.device, dtype=torch.float16)
    ids = captioner.generate(input_ids=entradas["input_ids"], pixel_values=entradas["pixel_values"],
                             max_new_tokens=20, num_beams=1, do_sample=False)
    descripciones = [t.strip() for t in procesador.batch_decode(ids, skip_special_tokens=True)]
    return [{"caja": [round(v, 1) for v in c], "descripcion": d} for c, d in zip(cajas, descripciones)]

In [ ]:
# prueba rápida: cajas y descripciones del formulario de alta
for el in ask_omniparser("02_alta_cliente.png"):
    print(el)

In [ ]:
# TODAS las cajas detectadas, numeradas -> vision_evaluation/omniparser/*__todas_las_cajas.png
_fuente = ImageFont.load_default(size=28)
cajas_omniparser = {c: ask_omniparser(c) for c in dict.fromkeys(e[0] for e in ELEMENTOS)}

for captura, elementos in cajas_omniparser.items():
    img = Image.open(CAPTURAS / captura).convert("RGB")
    draw = ImageDraw.Draw(img)
    for i, el in enumerate(elementos):
        x1, y1, x2, y2 = el["caja"]
        draw.rectangle((x1, y1, x2, y2), outline=(30, 120, 255), width=4)
        draw.text((x1 + 4, max(y1 - 32, 0)), str(i), fill=(30, 120, 255), font=_fuente)
    ruta = SALIDA / "omniparser" / f"{Path(captura).stem}__todas_las_cajas.png"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    img.save(ruta)
    print(f"{captura}: {len(elementos)} cajas -> {ruta}")

In [ ]:
# quedarse con la caja de cada elemento buscando su nombre en la descripción de Florence-2
# -> vision_evaluation/omniparser/
NOMBRE_OMNIPARSER = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre",
    "nif_field": "nif",
    "anadir_button": "anadir",
    "popup_aceptar_button": "aceptar",
}


def punto_omniparser(elementos: list[dict], nombre: str):
    """Centro de la primera caja cuya descripción contiene `nombre`, o None."""
    for el in elementos:
        if nombre in _normalizar(el["descripcion"]):
            x1, y1, x2, y2 = el["caja"]
            return (x1 + x2) / 2, (y1 + y2) / 2
    return None


for captura, elemento, _desc in ELEMENTOS:
    dibujar_comparacion("omniparser", captura, elemento,
                        punto_omniparser(cajas_omniparser[captura], NOMBRE_OMNIPARSER[elemento]))

## 3. UI-TARS-7B-DPO

`ByteDance-Seed/UI-TARS-7B-DPO` (~16 GB en bf16, arquitectura Qwen2-VL) está hecho para **agentes
que manejan interfaces**: le pides una acción en lenguaje natural ("Click on …") y responde
`click(start_box='(x,y)')` con las coordenadas de **ese** elemento.

**Las coordenadas van en escala 0-1000 sobre la imagen original**
([guía oficial](https://github.com/bytedance/UI-TARS/blob/main/README_v1.md#explanation-of-inference-results)):
`x_pixeles = x / 1000 * ancho`, `y_pixeles = y / 1000 * alto`.

In [ ]:
import re

_uitars = _uitars_procesador = None


def _cargar_uitars():
    global _uitars, _uitars_procesador
    if _uitars is None:
        from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

        modelo = "ByteDance-Seed/UI-TARS-7B-DPO"  # ~16 GB la primera vez
        _uitars_procesador = AutoProcessor.from_pretrained(modelo)
        _uitars = Qwen2VLForConditionalGeneration.from_pretrained(
            modelo, torch_dtype=torch.bfloat16, device_map="auto"
        )
    return _uitars, _uitars_procesador


def ask_uitars(captura: str, descripcion: str) -> str:
    """Respuesta en bruto de UI-TARS, p.ej. "click(start_box='(163,179)')"."""
    modelo, procesador = _cargar_uitars()
    imagen = Image.open(CAPTURAS / captura).convert("RGB")
    mensajes = [{"role": "user", "content": [
        {"type": "image", "image": imagen},
        {"type": "text", "text": f"Click on {descripcion}. Respond with click(start_box='(x,y)')."},
    ]}]
    texto = procesador.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = procesador(text=[texto], images=[imagen], return_tensors="pt").to(modelo.device)
    ids = modelo.generate(**entradas, max_new_tokens=128)
    return procesador.batch_decode(ids[:, entradas["input_ids"].shape[1]:], skip_special_tokens=True)[0]


def parse_uitars_xy(raw: str) -> tuple[float, float]:
    """(x, y) en escala 0-1000 sacados de click(start_box='(x,y)')."""
    m = re.search(r"start_box='?\(?\s*(-?\d+)\s*,\s*(-?\d+)\s*\)?'?", raw)
    if m:
        return float(m.group(1)), float(m.group(2))
    nums = re.findall(r"-?\d+\.?\d*", raw)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    raise ValueError(f"no pude extraer coordenadas de: {raw!r}")


def punto_uitars(captura: str, descripcion: str):
    """Coordenadas en píxeles de la captura, o None si la respuesta no trae coordenadas."""
    try:
        x, y = parse_uitars_xy(ask_uitars(captura, descripcion))
    except ValueError:
        return None
    ancho, alto = Image.open(CAPTURAS / captura).size
    return x / 1000 * ancho, y / 1000 * alto

In [ ]:
# prueba rápida con un solo elemento (el botón Entrar)
captura, _elemento, descripcion = ELEMENTOS[2]
print(ask_uitars(captura, descripcion))

In [ ]:
# imágenes etiquetadas -> vision_evaluation/uitars/
for captura, elemento, descripcion in ELEMENTOS:
    dibujar_comparacion("uitars", captura, elemento, punto_uitars(captura, descripcion))

## 4. Conclusiones

| Modelo | Qué hace bien | Su límite | Veredicto |
|---|---|---|---|
| **RapidOCR** | Localiza muy bien lo que **tiene texto**: botones, etiquetas. | **No ve los huecos vacíos**: en un campo para rellenar no hay texto, así que solo encuentra la etiqueta de al lado. No sirve para saber dónde escribir. | ❌ |
| **OmniParser** | El detector **encuentra bien todos los elementos**, incluso los campos vacíos (ver `*__todas_las_cajas.png`). | **Te los devuelve todos.** Para quedarte con el campo concreto hay que cribar, y la descripción de Florence-2 no lo resuelve (describe fotos, no botones en español). Montar ese filtro es una movida. | ❌ |
| **UI-TARS** | Le dices **qué buscas** y te da directamente **las coordenadas de ese elemento**. Además **responde preguntas sobre la imagen**. | Pesa ~16 GB y necesita GPU. | ✅ **Elegido** |

**Por qué UI-TARS:** era justo lo que queríamos: encontrar **el punto concreto** de un elemento a
partir de su descripción, **sin tener que cribar**. Y que además responda preguntas sobre la
pantalla nos sirvió después para que el agente **verifique** lo que ha escrito en cada campo.

Lo que salió de aquí se convirtió en `experiments/vision_tools.py` y, después, en `agente/vision.py`.